# Genie V1

在这里，我们将手写 [Genie](https://arxiv.org/abs/2402.15391) 的代码，理解其中的逻辑。当然，我们没有 Google 的数据集，也没有那么多的算力。因此，我们将仅聚焦于代码的书写而非将 Genie 真的造出来。

不过，有条件的情况下，可以考虑针对某个小数据集进行训练推理，体验这种生成式 AI。

这里用一张简图回顾下 Genie 的架构：

![Genie Architecture](assets/genie_arch.svg)

## 1. Video Tokenizer

这里，我们将写好 Video Tokenizer 的架构并进行训练。

### 1.1 Architecture

我之前在一篇文章中这么描述 Video Tokenizer:

> Video Tokenizer 整体上是一个 VQ-VAE 的架构。VQ-VAE 的核心作用是将连续的图像或视频块映射为离散的离散字典编码，从而将视觉生成转化为类似语言模型的 Token 预测任务。但与传统的 VQ-VAE 不同，文章使用 ST-Transformer 去实现 Encoder 和 Decoder，这样可以保障视频生成内容的连贯性以及可控的计算量。

似乎很简单。但实现起来（对我来说）非常复杂。首先需要进行 ST-Transformer 的实现，然后再进行 VAE 架构的魔改。

之前单独开了一篇文章实现了一个 encoder-only 的 ST-Transformer，现在可以直接来实现 VQ-VAE 了。

先来看看整体结构吧：

<div align="center">
  <img src="assets/genie_video_tokenizer.png" width="800">
</div>

### 1.2 Implementation

上图中的 Conv Encoder 与 ST-Transformer Encoder 是合并在一起做的，如下（和之前那篇文章很像）：

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
class STBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4):
        # mlp_ratio 控制隐藏层相对于输入维度大小的比值

        super().__init__()

        self.spatial_norm = nn.LayerNorm(dim)
        self.spatial_attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)

        self.temporal_norm = nn.LayerNorm(dim)
        self.temporal_attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)

        self.norm_ff = nn.LayerNorm(dim)

        self.ff = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio),
            nn.GELU(),
            nn.Linear(dim * mlp_ratio, dim),
        )

    def forward(self, x):
        # x: (B, T, H, W, D)

        B, T, H, W, D = x.shape

        # --- spatial ---
        xs = self.spatial_norm(x)
        xs = xs.reshape(B * T, H * W, D)
        xs, _ = self.spatial_attn(xs, xs, xs, need_weights=False) # self attention
        xs = xs.reshape(B, T, H, W, D)
        x = x + xs

        # --- temporal ---
        xt = self.temporal_norm(x)
        xt = xt.permute(0, 2, 3, 1, 4).reshape(B * H * W, T, D)
        xt, _ = self.temporal_attn(xt, xt, xt, need_weights=False)
        xt = xt.reshape(B, H, W, T, D).permute(0, 3, 1, 2, 4)
        x = x + xt

        # --- FFN ---
        x = x + self.ff(self.norm_ff(x))

        return x

In [4]:
class VideoEncoder(nn.Module):
    def __init__(
        self, 
        in_channels=3, 
        dim=256, 
        image_size=64,
        patch_size=8,
        num_frames=16,
        num_blocks=4, 
        num_heads=8,
    ):
        # input: (B, T, C, H_, W_) 
        super().__init__()

        self.dim = dim
        self.patch_size = patch_size

        assert image_size % patch_size == 0
        H = image_size // patch_size
        W = image_size // patch_size

        self.H = H
        self.W = W
        self.num_frames = num_frames

        # patch embedding
        # Compress image to prevent an explosion in computational complexity.
        self.patch_embed = nn.Conv2d(in_channels, dim, kernel_size=patch_size, stride=patch_size)

        # positional embeddings (learnable)
        self.temporal_pos = nn.Parameter(torch.randn(1, num_frames, 1, 1, dim) * 0.02)
        self.spatial_pos = nn.Parameter(torch.randn(1, 1, H, W, dim) * 0.02)

        # ST-Transformer blocks
        self.blocks = nn.ModuleList([
            STBlock(dim=dim, num_heads=num_heads)
            for _ in range(num_blocks)
        ])
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        """
        Args:
            x: raw pictures of (B, T, C, H_, W_).

        Returns:
            z_e: encoded pictures of (B, T, H, W, D)
        """

        B, T, C, H_, W_ = x.shape
        assert T <= self.num_frames
        x = x.reshape(B * T, C, H_, W_)

        # (B * T, D, H, W)
        x = self.patch_embed(x) 
        _, D, H, W = x.shape

        # (B, T, H, W, D)
        x = x.reshape(B, T, D, H, W).permute(0, 1, 3, 4, 2)
        
        x = x + self.temporal_pos[:, :T] + self.spatial_pos[:, :, :H, :W]

        for block in self.blocks:
            x = block(x)
        z_e = self.norm(x) # (B, T, H, W, D)

        return z_e

接下来是 Vector Quantization。先说说大致的实现思路。

其实就很像一个 K-means 聚类，我们从 encoder 拿到一个 $(B, T, H, W, D)$ 的向量后，对于每张编码后图像 $(H, W)$ 上的 $D$ 维向量，都与 Codebook 中的向量进行一一比较，找到最近的一个向量，并用那个向量所处的编号替换，这样就将 $(B, T, H, W, D)$ 的向量变成一堆 $(B, T, H, W)$ 的 token 序列。

可以形式化一下，记 encoder 输出的某个 $D$ 维向量为 $z_e$，codebook 包含的向量为 $\{e_1,\cdots,e_n\}$，则：

$$
k^*=\arg\min_k\left\|z_e-e_k\right\|_2^2
$$

输入 decoder 时，就查询 codebook 找到对应的向量就可以了。

这里存在一个问题，执行的 $\arg\min$ 这个操作本身是不可求导的。所以在实际训练的时候，我们在前向传播阶段正常使用 codebook，即 $z_q = e_{k^*}$；而在反向传播阶段，我们假定 VQ 层不存在，直接跳过。在代码中大致是这个样子：

```python
z_quantized = z_e + (z_q - z_e).detach()
```

不过这样的话，另一个问题又产生了：VQ 中没有梯度更新，我们无法训练 codebook。故而，我们需要额外增加两个损失项。

第一个是：

$$
\mathcal L_{\text{codebook}}=\left\|\operatorname{sg}[z_e]-e\right\|^2
$$

使得 $e$ 尽量跟进 encoder 的输出。

第二个是：

$$
\mathcal L_{\text{commit}}=\beta\left\|z_e-\operatorname{sg}[e]\right\|^2
$$

使得 encoder 也要尽量跟紧 codebook。

接下来，我们用代码去实现这个 VQ 层。（AI 样板 + 手工修改）

In [5]:
class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, beta=0.25):
        super().__init__()

        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.beta = beta

        self.codebook = nn.Embedding(num_embeddings, embedding_dim)

    def forward(self, z_e):
        """
        Args:
            z_e: Encoder features of shape (B * T, H, W, D).

        Returns:
            z_q: Quantized features of shape (B * T, H, W, D).
            indices: Discrete codebook indices of shape (B * T, H, W).
            vq_loss: Scalar vector-quantization loss.
        """

        BT, H, W, D = z_e.shape

        # -> (B*T*H*W, D)
        z_flattened = z_e.view(-1, D)

        # codebook: (K, D)
        embeddings = self.codebook.weight

        # distances:
        # (BTHW, K)
        distances = (
            z_flattened.pow(2).sum(dim=1, keepdim=True)
            + embeddings.pow(2).sum(dim=1)
            - 2 * z_flattened @ embeddings.T
        )

        indices = torch.argmin(distances, dim=1)
        quantized = self.codebook(indices).view(BT, H, W, D)

        codebook_loss = F.mse_loss(quantized, z_e.detach())
        commitment_loss = F.mse_loss(z_e, quantized.detach())
        loss = codebook_loss + self.beta * commitment_loss

        # straight-through estimator
        quantized = z_e + (quantized - z_e).detach()

        indices = indices.view(BT, H, W)

        return quantized, indices, loss

这里有一个工程细节，在计算 $\arg\min\left\|z_e-e_k\right\|^2$ 时，我们将其中的平方项展开：

$$
\|z_e - e_k \|^2 = \|z_e\|^2 + \|e_k\|^2 - 2 z_e^t e_k,
$$

这样可以避免张量相减时需要人为扩展成一个大张量。

之后我们马上来到 decoder 部分：

In [6]:
class VideoDecoder(nn.Module):
    def __init__(
        self,
        out_channels=3,
        dim=256,
        image_size=64,
        patch_size=8,
        num_frames=16,
        num_blocks=4,
        num_heads=8,
    ):
        super().__init__()

        self.dim = dim
        self.patch_size = patch_size

        assert image_size % patch_size == 0
        H = image_size // patch_size
        W = image_size // patch_size

        self.H = H
        self.W = W
        self.num_frames = num_frames

        # positional embeddings (learnable)
        self.temporal_pos = nn.Parameter(torch.randn(1, num_frames, 1, 1, dim) * 0.02)
        self.spatial_pos = nn.Parameter(torch.randn(1, 1, H, W, dim) * 0.02)

        # ST-Transformer blocks
        self.blocks = nn.ModuleList([
            STBlock(dim=dim, num_heads=num_heads)
            for _ in range(num_blocks)
        ])
        self.norm = nn.LayerNorm(dim)

        self.patch_unembed = nn.ConvTranspose2d(
            in_channels=dim,
            out_channels=out_channels,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, z_q):
        """
        Args:
            z_q: Quantized latent features of shape (B, T, H, W, D).

        Returns:
            x_hat: Reconstructed video of shape (B, T, C, H_, W_).
        """

        B, T, H, W, D = z_q.shape

        assert T <= self.num_frames
        assert H <= self.H
        assert W <= self.W
        assert D == self.dim

        x = z_q + self.temporal_pos[:, :T] + self.spatial_pos[:, :, :H, :W]

        for block in self.blocks:
            x = block(x)
        x = self.norm(x)

        # (B, T, H, W, D) -> (B * T, D, H, W)
        x = x.permute(0, 1, 4, 2, 3).contiguous()
        x = x.reshape(B * T, D, H, W)

        # (B * T, D, H, W) -> (B * T, C, H_, W_)
        x = self.patch_unembed(x)

        _, C, H_out, W_out = x.shape

        # -> (B, T, C, H_, W_)
        x_hat = x.reshape(B, T, C, H_out, W_out)

        return x_hat

基本框架完成，我们来简单测试一下一个随机数据能不能跑起来吧。

In [7]:
video_encoder = VideoEncoder()
video_decoder = VideoDecoder()
vector_quantizer = VectorQuantizer(8, 256)

B, T, C, H_, W_ = 8, 4, 3, 64, 64

x_test = torch.randn(B, T, C, H_, W_)
x_e = video_encoder(x_test)
print("x_e:", x_e.shape)
B, T, H, W, D = x_e.shape
x_e_flattened = x_e.reshape(B * T, H, W, D)
print("x_e_flattened:", x_e_flattened.shape)
x_q, _, _ = vector_quantizer(x_e_flattened)
x_q = x_q.reshape(B, T, H, W, D)
print("x_q:", x_q.shape)
x_hat = video_decoder(x_q)
print("x_hat:", x_hat.shape)

x_e: torch.Size([8, 4, 8, 8, 256])
x_e_flattened: torch.Size([32, 8, 8, 256])
x_q: torch.Size([8, 4, 8, 8, 256])
x_hat: torch.Size([8, 4, 3, 64, 64])


OK，检验完成，今天就到这儿吧。